In [13]:
!pip install beautifulsoup4 lxml pypdf pandas

In [14]:
from pathlib import Path
import json
import re
from bs4 import BeautifulSoup
from pypdf import PdfReader
import pandas as pd

def find_ai_lab_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if candidate.name == "ai_lab":
            return candidate
    raise RuntimeError("Không tìm thấy thư mục ai_lab. Hãy mở notebook bên trong project homelab.")

AI_LAB_ROOT = find_ai_lab_root(Path.cwd())
RAW_DIR = AI_LAB_ROOT / "raw"
EXTRACTED_DIR = AI_LAB_ROOT / "extracted"
NORMALIZED_DIR = AI_LAB_ROOT / "normalized"

EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

RAW_MANIFEST_PATH = RAW_DIR / "raw_manifest.jsonl"
EXTRACT_MANIFEST_PATH = EXTRACTED_DIR / "extract_manifest.jsonl"
NORMALIZED_DOCS_PATH = NORMALIZED_DIR / "docs.jsonl"

print("AI_LAB_ROOT =", AI_LAB_ROOT)
print("RAW_MANIFEST_PATH =", RAW_MANIFEST_PATH)

AI_LAB_ROOT = D:\homelab\ai_lab
RAW_MANIFEST_PATH = D:\homelab\ai_lab\raw\raw_manifest.jsonl


In [15]:
def read_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

records = read_jsonl(RAW_MANIFEST_PATH)
print(f"Số nguồn trong raw_manifest: {len(records)}")
pd.DataFrame(records)

Số nguồn trong raw_manifest: 8


,source_id,source_name,source_url,local_path,doc_type,language,section_target,priority,review_required,use_in_v1,status
0,blood_tests,NHS,https://www.nhs.uk/tests-and-treatments/blood-...,ai_lab/raw/blood_tests/source.html,html,en,"[test_explainers, pre_test_guides]",high,False,True,ok
1,chest_pain,NHS,https://www.nhs.uk/symptoms/chest-pain/,ai_lab/raw/chest_pain/source.html,html,en,[red_flags],high,True,True,ok
2,shortness_of_breath,NHS,https://www.nhs.uk/symptoms/shortness-of-breath/,ai_lab/raw/shortness_of_breath/source.html,html,en,[red_flags],high,True,True,ok
3,cdc_dpdx_blood_collection,CDC,https://www.cdc.gov/dpdx/diagnosticprocedures/...,ai_lab/raw/cdc_dpdx_blood_collection/source.html,html,en,[sample_collection],low,True,False,ok
4,cdc_specimen_packing_and_shipping,CDC,https://www.cdc.gov/laboratory/specimen-submis...,ai_lab/raw/cdc_specimen_packing_and_shipping/s...,pdf,en,[lab_ops],low,True,False,ok
5,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,ai_lab/raw/nice_sepsis_overview/source.pdf,pdf,en,[red_flags],medium,True,True,ok
6,nice_sepsis_guideline,NICE,https://www.nice.org.uk/guidance/ng253/resourc...,ai_lab/raw/nice_sepsis_guideline/source.html,html,en,[red_flags],medium,True,True,ok
7,who_infectious_shipping_guidance,WHO,https://cdn.who.int/media/docs/default-source/...,ai_lab/raw/who_infectious_shipping_guidance/so...,pdf,en,[lab_ops],low,True,False,ok


In [16]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [17]:
def extract_html_text(html_path: Path) -> dict:
    with open(html_path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    soup = BeautifulSoup(html, "lxml")

    # bỏ tag rác
    for tag in soup(["script", "style", "noscript", "svg", "header", "footer", "nav", "form", "aside"]):
        tag.decompose()

    # ưu tiên article/main
    main_node = (
        soup.find("article")
        or soup.find("main")
        or soup.find(attrs={"role": "main"})
        or soup.body
    )

    title = ""
    if soup.title and soup.title.text:
        title = clean_text(soup.title.text)

    body_text = clean_text(main_node.get_text("\n", strip=True)) if main_node else ""

    return {
        "title": title,
        "content": body_text
    }

In [18]:
def extract_pdf_text(pdf_path: Path) -> dict:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        try:
            txt = page.extract_text() or ""
        except Exception:
            txt = ""
        pages.append(txt)

    content = clean_text("\n\n".join(pages))
    title = pdf_path.stem

    return {
        "title": title,
        "content": content
    }

In [19]:
def infer_section(section_target):
    if not section_target:
        return "general"
    return section_target[0]

def infer_risk_level(section_target):
    if "red_flags" in section_target:
        return "high"
    if "sample_collection" in section_target:
        return "medium"
    return "low"

def build_normalized_doc(record, extracted):
    return {
        "doc_id": f"{record['source_id']}_doc_001",
        "source_id": record["source_id"],
        "source_name": record["source_name"],
        "source_url": record["source_url"],
        "local_path": record["local_path"],
        "doc_type": record["doc_type"],
        "language": record["language"],
        "section": infer_section(record.get("section_target", [])),
        "section_target": record.get("section_target", []),
        "title": extracted["title"] or record["source_id"],
        "content": extracted["content"],
        "priority": record.get("priority", "medium"),
        "review_required": record.get("review_required", True),
        "use_in_v1": record.get("use_in_v1", False),
        "risk_level": infer_risk_level(record.get("section_target", [])),
        "review_status": "pending"
    }

In [20]:
extract_logs = []
normalized_docs = []

for record in records:
    rel_path = record["local_path"]
    file_path = AI_LAB_ROOT.parent / rel_path
    
    if not file_path.exists():
        extract_logs.append({
            "source_id": record["source_id"],
            "input_file": str(file_path),
            "output_file": None,
            "status": "missing_file",
            "char_count": 0
        })
        continue

    try:
        if record["doc_type"] == "html":
            extracted = extract_html_text(file_path)
            out_path = EXTRACTED_DIR / f"{record['source_id']}.txt"
        elif record["doc_type"] == "pdf":
            extracted = extract_pdf_text(file_path)
            out_path = EXTRACTED_DIR / f"{record['source_id']}.txt"
        else:
            raise ValueError(f"doc_type không hỗ trợ: {record['doc_type']}")

        with open(out_path, "w", encoding="utf-8") as f:
            f.write(extracted["content"])

        extract_logs.append({
            "source_id": record["source_id"],
            "input_file": str(file_path),
            "output_file": str(out_path),
            "status": "ok",
            "char_count": len(extracted["content"])
        })

        normalized_docs.append(build_normalized_doc(record, extracted))

    except Exception as e:
        extract_logs.append({
            "source_id": record["source_id"],
            "input_file": str(file_path),
            "output_file": None,
            "status": f"error: {str(e)}",
            "char_count": 0
        })

print("Extract xong.")
print("Số doc normalized:", len(normalized_docs))
pd.DataFrame(extract_logs)

Extract xong.
Số doc normalized: 8


,source_id,input_file,output_file,status,char_count
0,blood_tests,D:\homelab\ai_lab\raw\blood_tests\source.html,D:\homelab\ai_lab\extracted\blood_tests.txt,ok,4097
1,chest_pain,D:\homelab\ai_lab\raw\chest_pain\source.html,D:\homelab\ai_lab\extracted\chest_pain.txt,ok,2459
2,shortness_of_breath,D:\homelab\ai_lab\raw\shortness_of_breath\sour...,D:\homelab\ai_lab\extracted\shortness_of_breat...,ok,3006
3,cdc_dpdx_blood_collection,D:\homelab\ai_lab\raw\cdc_dpdx_blood_collectio...,D:\homelab\ai_lab\extracted\cdc_dpdx_blood_col...,ok,2945
4,cdc_specimen_packing_and_shipping,D:\homelab\ai_lab\raw\cdc_specimen_packing_and...,D:\homelab\ai_lab\extracted\cdc_specimen_packi...,ok,8613
5,nice_sepsis_overview,D:\homelab\ai_lab\raw\nice_sepsis_overview\sou...,D:\homelab\ai_lab\extracted\nice_sepsis_overvi...,ok,128225
6,nice_sepsis_guideline,D:\homelab\ai_lab\raw\nice_sepsis_guideline\so...,D:\homelab\ai_lab\extracted\nice_sepsis_guidel...,ok,3830
7,who_infectious_shipping_guidance,D:\homelab\ai_lab\raw\who_infectious_shipping_...,D:\homelab\ai_lab\extracted\who_infectious_shi...,ok,135741


In [21]:
with open(EXTRACT_MANIFEST_PATH, "w", encoding="utf-8") as f:
    for row in extract_logs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Đã ghi:", EXTRACT_MANIFEST_PATH)

Đã ghi: D:\homelab\ai_lab\extracted\extract_manifest.jsonl


In [22]:
with open(NORMALIZED_DOCS_PATH, "w", encoding="utf-8") as f:
    for row in normalized_docs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Đã ghi:", NORMALIZED_DOCS_PATH)

Đã ghi: D:\homelab\ai_lab\normalized\docs.jsonl


In [23]:
df_docs = pd.DataFrame(normalized_docs)
df_docs[["source_id", "title", "section", "risk_level", "use_in_v1"]]

,source_id,title,section,risk_level,use_in_v1
0,blood_tests,Blood tests\n - NHS,test_explainers,low,True
1,chest_pain,Chest pain\n - NHS,red_flags,high,True
2,shortness_of_breath,Shortness of breath\n - NHS,red_flags,high,True
3,cdc_dpdx_blood_collection,CDC - DPDx - Diagnostic Procedures - Blood Spe...,sample_collection,medium,False
4,cdc_specimen_packing_and_shipping,source,lab_ops,low,False
5,nice_sepsis_overview,source,red_flags,high,True
6,nice_sepsis_guideline,Overview | Suspected sepsis in people aged 16 ...,red_flags,high,True
7,who_infectious_shipping_guidance,source,lab_ops,low,False


In [24]:
for row in normalized_docs[:3]:
    print("=" * 80)
    print("SOURCE:", row["source_id"])
    print("TITLE :", row["title"])
    print("SECTION:", row["section"])
    print("CONTENT PREVIEW:\n")
    print(row["content"][:1500])
    print("\n")

SOURCE: blood_tests
TITLE : Blood tests
 - NHS
SECTION: test_explainers
CONTENT PREVIEW:

A blood test is often done to check your health, or to find out why you're having certain symptoms. It involves having a small amount of your blood taken for testing.
Why a blood test is done
There are lots of reasons why you may need a blood test.
A blood test may be done to:
check your general health
find out if symptoms you're having are caused by certain conditions
find out if you're more likely to get a condition
find out how well a condition is being treated or managed
How to book a blood test
If a healthcare professional such as a GP, nurse or specialist thinks you need a blood test they will tell you how to book one.
Some GP surgeries or hospitals allow you to book a blood test online.
Types of blood test
Blood tests can check for different things depending on your symptoms, any conditions you have, and any medicines you’re taking.
Common types of blood test
Test
Why it's done
Full blood c